# Baseline Model with k-fold Cross Validation

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [2]:
# Import necessary libraries

import random

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from datasets import concatenate_datasets, load_dataset


In [3]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")   # Apple Silicon (M1/M2/M3)
else:
    DEVICE = torch.device("cpu")   # Fallback

print(f"Using Device: {DEVICE}")

Using Device: mps


## Model Choice

[Explain why you've chosen a particular model as the baseline. This could be a simple statistical model or a basic machine learning model. Justify your choice.]


We chose a convolutional neural network (CNN) as the baseline model for our aesthetic emotions map project. CNNs are well-suited for image classification tasks due to their ability to capture spatial hierarchies in images. They can learn to recognize patterns and features in the images that are relevant for predicting the associated emotions. Additionally, CNNs have been widely used and have shown strong performance in various image-related tasks, making them a reasonable starting point for our project.

## Feature Selection

[Indicate which features from the dataset you will be using for the baseline model, and justify your selection.]

For the baseline model, we will be using the raw pixel values of the images as features. This is a common approach for image classification tasks, as it allows the model to learn directly from the visual data without any manual feature engineering. By using the raw pixel values, we can leverage the CNN's ability to automatically extract relevant features from the images during training.

In [4]:
# Loading the dataset using Hugging Face's datasets library
dataset = load_dataset("bjoern-doege/aesthetic-emotions-map")

Resolving data files:   0%|          | 0/4721 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1025 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1089 [00:00<?, ?it/s]

In [5]:
NORMALIZE_MEAN=[0.5111283659934998, 0.48830345273017883, 0.46479079127311707]
NORMALIZE_STD=[0.3433663249015808, 0.3207928538322449, 0.32255250215530396]

In [6]:
BASE_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

In [7]:
# Converting labels to ids
# PyTorch's ImageFolder would do this automatically, but we are using Hugging Face's datasets library
label_names = sorted(dataset["train"].unique("label"))
label_to_id = {label: i for i, label in enumerate(label_names)}

In [8]:
# This function will be applied to each example in the dataset to preprocess the images and labels
# Again, this is necessary because we are using Hugging Face's datasets library instead of PyTorch's ImageFolder
def preprocess(examples):
    return {
        "image": [
            BASE_TRANSFORM(image.convert("RGB"))
            for image in examples["image"]
        ],
        "label": torch.tensor(
            [label_to_id[label] for label in examples["label"]],
            dtype=torch.long,
        ),
    }

In [9]:
# Use train + validation for cross-validation and keep test untouched for final evaluation.
trainval_dataset = concatenate_datasets([
    dataset["train"],
    dataset["validation"],
])

test_dataset = dataset["test"]

BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
N_SPLITS = 5
RANDOM_STATE = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(RANDOM_STATE)


In [10]:
X = np.arange(len(trainval_dataset))
y = [label_to_id[label] for label in trainval_dataset["label"]]
groups = trainval_dataset["style"]

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

style_overlap = set(trainval_dataset["style"]) & set(test_dataset["style"])
print(f"Train+validation samples: {len(trainval_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Overlapping styles between train+validation and test: {len(style_overlap)}")


Train+validation samples: 5744
Test samples: 1088
Overlapping styles between train+validation and test: 0


## Implementation

[Implement your baseline model here.]

The baseline CNN architecture consists of the following layers assuming an input image size of 224x224 pixels:
1. **Convolutional Layer 1**: 32 filters, kernel size of 3x3, ReLU activation
2. **Max Pooling Layer 1**: Pool size of 2x2
3. **Convolutional Layer 2**: 64 filters, kernel size of 3x3, ReLU activation
4. **Max Pooling Layer 2**: Pool size of 2x2
5. **Convolutional Layer 3**: 128 filters, kernel size of 3x3, ReLU activation
6. **Max Pooling Layer 3**: Pool size of 2x2
7. **Flatten Layer**: Flattens the output from the previous layer
8. **Fully Connected Layer 1**: 256 neurons, ReLU activation
9. **Dropout Layer**: Dropout rate of 0.5 to reduce overfitting
10. **Output Layer**: Number of neurons equal to the number of emotion classes

In [11]:
# Initialize and train the baseline model
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Block 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # For 224x224 input images: 224 -> 112 -> 56 -> 28
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [12]:
def create_model():
    return BaselineCNN(num_classes=len(label_names)).to(DEVICE)


def create_optimizer(model):
    return optim.Adam(model.parameters(), lr=LEARNING_RATE)


def create_loss_fn():
    return nn.CrossEntropyLoss()


def create_dataloader(ds, batch_size=BATCH_SIZE, shuffle=False, seed=RANDOM_STATE):
    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(seed)

    return DataLoader(
        ds.with_transform(preprocess),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )


In [13]:
def train_one_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    return compute_metrics(y_true, y_pred, running_loss)


def evaluate(model, data_loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in data_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            running_loss += loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    return compute_metrics(y_true, y_pred, running_loss)


def compute_metrics(y_true, y_pred, total_loss):
    return {
        "loss": total_loss / len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }


In [14]:
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups), start=1):
    print(f"\nFold {fold}/{N_SPLITS}")
    fold_seed = RANDOM_STATE + fold
    seed_everything(fold_seed)

    fold_train_dataset = trainval_dataset.select(train_idx.tolist())
    fold_val_dataset = trainval_dataset.select(val_idx.tolist())

    train_loader = create_dataloader(fold_train_dataset, shuffle=True, seed=fold_seed)
    val_loader = create_dataloader(fold_val_dataset, shuffle=False)

    model = create_model()
    optimizer = create_optimizer(model)
    loss_fn = create_loss_fn()

    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_metrics = evaluate(model, val_loader, loss_fn, DEVICE)

        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
            f"train_loss={train_metrics['loss']:.4f}, "
            f"train_acc={train_metrics['accuracy']:.4f}, "
            f"train_macro_f1={train_metrics['f1_macro']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}, "
            f"val_acc={val_metrics['accuracy']:.4f}, "
            f"val_macro_f1={val_metrics['f1_macro']:.4f}"
        )

        fold_metrics.append({
            "fold": fold,
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1_weighted": train_metrics["f1_weighted"],
            "train_f1_macro": train_metrics["f1_macro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_weighted": val_metrics["f1_weighted"],
            "val_f1_macro": val_metrics["f1_macro"],
        })

    """ fold_metrics.append({
        "fold": fold,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "val_f1_macro": val_metrics["f1_macro"],
    }) """

val_macro_f1_scores = [metrics["val_f1_macro"] for metrics in fold_metrics]
val_accuracy_scores = [metrics["val_accuracy"] for metrics in fold_metrics]

print("\nCross-validation summary")
print(f"Accuracy: {np.mean(val_accuracy_scores):.4f} +/- {np.std(val_accuracy_scores):.4f}")
print(f"Macro F1: {np.mean(val_macro_f1_scores):.4f} +/- {np.std(val_macro_f1_scores):.4f}")



Fold 1/5
Epoch 1/10 | train_loss=1.7987, train_acc=0.3232, train_macro_f1=0.2514 | val_loss=1.8255, val_acc=0.3672, val_macro_f1=0.2323
Epoch 2/10 | train_loss=1.5070, train_acc=0.4207, train_macro_f1=0.3819 | val_loss=1.9737, val_acc=0.3498, val_macro_f1=0.2261
Epoch 3/10 | train_loss=1.3438, train_acc=0.4911, train_macro_f1=0.4603 | val_loss=1.9146, val_acc=0.3828, val_macro_f1=0.2726
Epoch 4/10 | train_loss=1.2044, train_acc=0.5507, train_macro_f1=0.5267 | val_loss=2.0211, val_acc=0.3802, val_macro_f1=0.2986
Epoch 5/10 | train_loss=1.0300, train_acc=0.6178, train_macro_f1=0.5980 | val_loss=2.1816, val_acc=0.3785, val_macro_f1=0.2674
Epoch 6/10 | train_loss=0.8500, train_acc=0.6908, train_macro_f1=0.6785 | val_loss=2.3207, val_acc=0.3785, val_macro_f1=0.2912
Epoch 7/10 | train_loss=0.6741, train_acc=0.7639, train_macro_f1=0.7558 | val_loss=2.7931, val_acc=0.3663, val_macro_f1=0.2872
Epoch 8/10 | train_loss=0.4698, train_acc=0.8351, train_macro_f1=0.8311 | val_loss=3.2972, val_acc=0.

## Evaluation

[Clearly state what metrics you will use to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.]

We are using the following metrics to evaluate the performance of our baseline model:
1. **Accuracy**: The proportion of correctly classified instances among the total instances.
2. **Precision**: The proportion of true positive predictions among all positive predictions.
3. **Recall**: The proportion of true positive predictions among all actual positives.
4. **F1 Score**: The harmonic mean of precision and recall, providing a balance between the two metrics.
5. **Confusion Matrix**: A table that describes the performance of the classification model by showing the true positives, true negatives, false positives, and false negatives for each class.
6. **Classification Report**: A comprehensive report that includes precision, recall, F1 score, and support for each class, providing a detailed overview of the model's performance across all classes.

In [15]:
eval_metrics = []

# Train one final model on all train+validation data, then evaluate once on the untouched test set.
final_seed = RANDOM_STATE + N_SPLITS + 1
seed_everything(final_seed)
final_model = create_model()
final_optimizer = create_optimizer(final_model)
final_loss_fn = create_loss_fn()

trainval_loader = create_dataloader(trainval_dataset, shuffle=True, seed=final_seed)
test_loader = create_dataloader(test_dataset, shuffle=False)

for epoch in range(NUM_EPOCHS):
    train_metrics = train_one_epoch(final_model, trainval_loader, final_optimizer, final_loss_fn, DEVICE)
    val_metrics = evaluate(final_model, test_loader, final_loss_fn, DEVICE)
    print(
        f"Final model epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f}, "
        f"train_acc={train_metrics['accuracy']:.4f}, "
        f"train_macro_f1={train_metrics['f1_macro']:.4f}, "
        f"val_loss={val_metrics['loss']:.4f}, "
        f"val_acc={val_metrics['accuracy']:.4f}, "
        f"val_macro_f1={val_metrics['f1_macro']:.4f}"
    )

    eval_metrics.append({
        "epoch": epoch + 1,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_f1_weighted": train_metrics["f1_weighted"],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "val_f1_macro": val_metrics["f1_macro"],
    })

test_metrics = evaluate(final_model, test_loader, final_loss_fn, DEVICE)

print("\nFinal test metrics")
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Weighted precision: {test_metrics['precision_weighted']:.4f}")
print(f"Weighted recall: {test_metrics['recall_weighted']:.4f}")
print(f"Weighted F1: {test_metrics['f1_weighted']:.4f}")
print(f"Macro F1: {test_metrics['f1_macro']:.4f}")

print("\nClassification report:")
print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0,
))

print("\nConfusion matrix:")
print(confusion_matrix(test_metrics["y_true"], test_metrics["y_pred"]))


Final model epoch 1/10 | train_loss=1.8091, train_acc=0.3339, train_macro_f1=0.2466, val_loss=1.6511, val_acc=0.3961, val_macro_f1=0.2866
Final model epoch 2/10 | train_loss=1.5099, train_acc=0.4239, train_macro_f1=0.3721, val_loss=1.5979, val_acc=0.4274, val_macro_f1=0.3506
Final model epoch 3/10 | train_loss=1.3527, train_acc=0.4920, train_macro_f1=0.4566, val_loss=1.6275, val_acc=0.4062, val_macro_f1=0.3362
Final model epoch 4/10 | train_loss=1.1696, train_acc=0.5562, train_macro_f1=0.5325, val_loss=1.6003, val_acc=0.4274, val_macro_f1=0.3810
Final model epoch 5/10 | train_loss=1.0023, train_acc=0.6290, train_macro_f1=0.6093, val_loss=1.9711, val_acc=0.3686, val_macro_f1=0.3153
Final model epoch 6/10 | train_loss=0.8313, train_acc=0.6995, train_macro_f1=0.6890, val_loss=1.9407, val_acc=0.4467, val_macro_f1=0.3858
Final model epoch 7/10 | train_loss=0.6516, train_acc=0.7641, train_macro_f1=0.7595, val_loss=2.0052, val_acc=0.4577, val_macro_f1=0.3863
Final model epoch 8/10 | train_los

In [16]:
torch.save(final_model.state_dict(), "model_w_k-fold.pth")


In [17]:
df_fold = pd.DataFrame(fold_metrics)
df_fold.to_csv("eval_fold_metrics_seed_42_w_k-fold.csv", index=False)

df_eval = pd.DataFrame(eval_metrics)
df_eval.to_csv("eval_eval_metrics_seed_42_w_k-fold.csv", index=False)